# Remote SSH validation for RaspTank
Select the intended Python kernel and open this repository in Remote SSH.
Discovery and dry-run checks also work locally without Raspberry Pi dependencies.
Real hardware access is confined to the disabled final cell. Do not import legacy server
modules for discovery: they can initialize hardware at import time.


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'rasptank' / '__init__.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open the repository or its notebooks directory first')
print('Repository:', repo_root, 'Python:', sys.executable)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
# Avoid dotted find_spec: it imports parents and can fail on incomplete branches.
required_paths = ('__init__.py', 'config.py', 'exceptions.py', 'tank.py',
                  'hardware/motors.py', 'hardware/servos.py', 'hardware/ultrasonic.py',
                  'hardware/leds.py', 'hardware/display.py')
for name in required_paths:
    print(name, 'FOUND' if (repo_root / 'rasptank' / name).is_file() else 'MISSING')


In [ ]:
import importlib
modules = ('rasptank', 'rasptank.config', 'rasptank.exceptions', 'rasptank.tank',
           'rasptank.hardware.motors', 'rasptank.hardware.servos',
           'rasptank.hardware.ultrasonic', 'rasptank.hardware.leds',
           'rasptank.hardware.display')
failures = []
for name in modules:
    try:
        importlib.import_module(name)
        print(name, 'IMPORT_OK')
    except (ImportError, RuntimeError) as exc:
        failures.append(name)
        print(name, type(exc).__name__, str(exc))
assert not failures, failures


In [ ]:
from rasptank import RaspTank
tank = RaspTank(use_gpio=False, enable_servos=True, enable_leds=True, enable_display=True)
try:
    for method in ('forward', 'backward', 'turn_left', 'turn_right'):
        getattr(tank, method)(15)
    tank.stop()
    assert tank.ultrasonic is None
    assert tank.motors.pwm_a is None and tank.servos.pwm is None
    tank.servos.set_pulse(1, 300)
    assert tank.servos.move_relative(1, 10) == 290
    tank.leds.off()
    tank.display.show('REMOTE', 'DRY RUN')
    print('SYMBOLIC_OK: no GPIO or I2C devices opened')
finally:
    tank.close()


## Manual hardware safety gate
Read README's manual Pi steps. Stop the legacy server; verify BCM wiring, power and
the remote Pi kernel. Secure the chassis on a stand with tracks clear and have a physical
power cutoff available. Low duty may stall a track and does not guarantee safe movement.
Only stopped motors are initialized below; optional devices remain off, and no servo is
centered implicitly. Software checks cannot establish physical direction or calibration.
Calibration in the new API is instance-local, never loaded from or written to legacy files.


In [ ]:
RUN_SAFE_TEST = False
if RUN_SAFE_TEST:
    import time
    hardware_tank = None
    try:
        hardware_tank = RaspTank(use_gpio=True)
        hardware_tank.forward(15)
        time.sleep(0.3)
        hardware_tank.stop()
    finally:
        if hardware_tank is not None:
            hardware_tank.close()
else:
    print('SAFE_TEST_DISABLED')
